In [ ]:
### Santiago Alvarez Tostado Estrada

In [ ]:
##### Este archivo contiene diferentes usos de python y pandas para diferentes situaciones en mi trabajo actual. Los hice con el fin de ahorrar tiempo y esfuerzo. Resultaron ser bastante efectivos ya que la tienda cuenta con mas de 6000 productos y estar buscando discrepancias uno por uno resultaria en

# Comparacion de precios a publico del sistema viejo a sistema nuevo

##### Originalmente la idea era cobrar con los dos sistemas a la vez e ir cambiando y modificando precios conforme vayan saliendo. Opte por una opcion mas rapida en la que se hacia unn crossmatch para sacar un reporte con todas las discrepancias. No fue nada facil debido al formato en el que venia uno de los dos archivos. los codigos numericos si debian de ser del mismo producto por lo que se separaron con los codigos alfanumericos. Los alfanumericos podrian variar y tener ligados diferentes productos, por los que a todos se les hizo un crossmatch usando la descripcion como eje. 

In [5]:
import pandas as pd
import numpy as np

ruta_pm1 = r"C:\LAB3\3\DATA\pm1.xlsx"
ruta_sicar = r"C:\LAB3\3\DATA\artSicar.xlsx"

print("Leyendo el archivo Sicar...")
# 2. Leer y extraer datos de Sicar desde formato Excel
sicar_raw = pd.read_excel(ruta_sicar, header=None)

sicar_data = []
desc = None
clave = None
precio = None

# Recorrer cada renglón del Excel de Sicar buscando la info
for index, row in sicar_raw.iterrows():
    col1 = str(row[1]).strip() if pd.notna(row[1]) else ""
    
    if col1 != "" and col1 not in ["Clave:", "Departamento:", "Categoría:", "EL MERCADITO DE MAMA"]:
        desc = col1
        
    if col1 == "Clave:":
        clave = str(row[3]).strip() if pd.notna(row[3]) else None
        
    if col1 == "Departamento:":
        try:
            precio_str = str(row[9]).replace('$', '').replace(',', '').strip()
            precio = float(precio_str)
        except:
            precio = np.nan
            
        if desc and clave:
            # Guardamos la descripción con un nombre específico para Sicar
            sicar_data.append({'Descripcion_SICAR': desc, 'Clave_sicar': clave, 'Precio_sicar': precio})
            
        desc = None
        clave = None
        precio = None

sicar_df = pd.DataFrame(sicar_data)
print(f"Productos extraídos de Sicar: {len(sicar_df)}")

print("Leyendo el archivo pm1...")
# 3. Leer pm1.xlsx inteligentemente
pm1_temp = pd.read_excel(ruta_pm1, header=None)

header_idx = 0
for i, row in pm1_temp.iterrows():
    row_str = " ".join([str(val) for val in row.values])
    if "Codigo barras" in row_str or "Clave" in row_str:
        header_idx = i
        break

pm1_df = pd.read_excel(ruta_pm1, header=header_idx)
pm1_df.columns = [str(col).strip() for col in pm1_df.columns]

# Renombrar la columna de descripción de PM1 para que sea clara
col_desc_pm1 = 'Descripcion' if 'Descripcion' in pm1_df.columns else pm1_df.columns[2]
pm1_df = pm1_df.rename(columns={col_desc_pm1: 'Descripcion PUNTO MI'})

# 4. Separar códigos numéricos y alfanuméricos
def is_numeric(x):
    if pd.isna(x): return False
    return str(x).strip().isnumeric()

col_codigo = 'Codigo barras' if 'Codigo barras' in pm1_df.columns else pm1_df.columns[3]
col_precio = 'Precio N° 1' if 'Precio N° 1' in pm1_df.columns else [c for c in pm1_df.columns if 'Precio' in c][0]

pm1_df['Codigo_is_numeric'] = pm1_df[col_codigo].apply(is_numeric)

pm1_num = pm1_df[pm1_df['Codigo_is_numeric']].copy()
pm1_alpha = pm1_df[~pm1_df['Codigo_is_numeric']].copy()

print("Cruzando información...")
# 5. Cruzar los datos numéricos por código de barras
pm1_num['Codigo barras_str'] = pm1_num[col_codigo].astype(str).str.strip()
sicar_df['Clave_sicar_str'] = sicar_df['Clave_sicar'].astype(str).str.strip()
merged_num = pd.merge(pm1_num, sicar_df, left_on='Codigo barras_str', right_on='Clave_sicar_str', how='left')

# 6. Cruzar los datos alfanuméricos por descripción
pm1_alpha['Desc_upper'] = pm1_alpha['Descripcion PUNTO MI'].astype(str).str.upper().str.strip()
sicar_df['Desc_upper'] = sicar_df['Descripcion_SICAR'].astype(str).str.upper().str.strip()
merged_alpha = pd.merge(pm1_alpha, sicar_df, left_on='Desc_upper', right_on='Desc_upper', how='left', suffixes=('', '_y'))

# 7. Unir los resultados y calcular la diferencia
res = pd.concat([merged_num, merged_alpha], ignore_index=True)
res['Diferencia_Precios'] = res['Precio_sicar'] - pd.to_numeric(res[col_precio], errors='coerce')

# Limpiar duplicados
res = res.drop_duplicates(subset=['Clave', col_codigo])

# 8. Filtrar solo los que tienen diferencia de precio
res_con_diferencia = res[res['Diferencia_Precios'].notna() & (res['Diferencia_Precios'] != 0)]

# 9. Guardar el archivo
# AQUÍ ESTÁN LAS DOS COLUMNAS DE DESCRIPCIÓN SOLICITADAS
out_cols = ['Clave', 'Descripcion PUNTO MI', 'Descripcion_SICAR', col_codigo, col_precio, 'Precio_sicar', 'Diferencia_Precios']
res_final = res_con_diferencia[out_cols]

ruta_salida = r"C:\LAB3\3\DATA\Diferencias.csv"
res_final.to_csv(ruta_salida, index=False, encoding="utf-8-sig")

print(f"Proceso terminado. Se encontraron {len(res_final)} artículos con diferencia de precio.")
print(f"El archivo se ha guardado exitosamente en: {ruta_salida}")

Leyendo el archivo Sicar...


C:\Users\alvar\anaconda3\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


Productos extraídos de Sicar: 6498
Leyendo el archivo pm1...
Cruzando información...
Proceso terminado. Se encontraron 182 artículos con diferencia de precio.
El archivo se ha guardado exitosamente en: C:\LAB3\3\DATA\Diferencias.csv


# Busqueda de productos faltantes

##### Usando los mismos datasets, se hizo una busqueda de productos que estaban en el sistema viejo y no en el nuevo. 

In [1]:
import pandas as pd
import numpy as np

# 1. Definir las rutas exactas
ruta_pm1 = r"C:\LAB3\3\DATA\pm1.xlsx"
ruta_sicar = r"C:\LAB3\3\DATA\artSicar.xlsx"

print("Leyendo el archivo Sicar...")
# 2. Leer y extraer datos de Sicar desde formato Excel
sicar_raw = pd.read_excel(ruta_sicar, header=None)

sicar_data = []
desc = None
clave = None
precio = None

for index, row in sicar_raw.iterrows():
    col1 = str(row[1]).strip() if pd.notna(row[1]) else ""
    
    if col1 != "" and col1 not in ["Clave:", "Departamento:", "Categoría:", "EL MERCADITO DE MAMA"]:
        desc = col1
        
    if col1 == "Clave:":
        clave = str(row[3]).strip() if pd.notna(row[3]) else None
        
    if col1 == "Departamento:":
        try:
            precio_str = str(row[9]).replace('$', '').replace(',', '').strip()
            precio = float(precio_str)
        except:
            precio = np.nan
            
        if desc and clave:
            sicar_data.append({'Descripcion_SICAR': desc, 'Clave_sicar': clave, 'Precio_sicar': precio})
            
        desc = None
        clave = None
        precio = None

sicar_df = pd.DataFrame(sicar_data)
print(f"Total de productos extraídos de Sicar: {len(sicar_df)}")

print("Leyendo el archivo pm1...")
# 3. Leer pm1.xlsx inteligentemente
pm1_temp = pd.read_excel(ruta_pm1, header=None)

header_idx = 0
for i, row in pm1_temp.iterrows():
    row_str = " ".join([str(val) for val in row.values])
    if "Codigo barras" in row_str or "Clave" in row_str:
        header_idx = i
        break

pm1_df = pd.read_excel(ruta_pm1, header=header_idx)
pm1_df.columns = [str(col).strip() for col in pm1_df.columns]

col_desc_pm1 = 'Descripcion' if 'Descripcion' in pm1_df.columns else pm1_df.columns[2]
col_codigo = 'Codigo barras' if 'Codigo barras' in pm1_df.columns else pm1_df.columns[3]

# 4. Separar códigos numéricos y alfanuméricos de pm1
def is_numeric(x):
    if pd.isna(x): return False
    return str(x).strip().isnumeric()

pm1_df['Codigo_is_numeric'] = pm1_df[col_codigo].apply(is_numeric)

pm1_num = pm1_df[pm1_df['Codigo_is_numeric']]
pm1_alpha = pm1_df[~pm1_df['Codigo_is_numeric']]

# 5. Crear listas de referencia rápidas para comparar
# Códigos numéricos de pm1
pm1_codigos_num = set(pm1_num[col_codigo].astype(str).str.strip())
# Descripciones de los que son alfanuméricos en pm1 (en mayúsculas)
pm1_desc_alpha = set(pm1_alpha[col_desc_pm1].astype(str).str.upper().str.strip())

# Preparar las columnas de Sicar para la comparación
sicar_df['Clave_sicar_str'] = sicar_df['Clave_sicar'].astype(str).str.strip()
sicar_df['Desc_upper'] = sicar_df['Descripcion_SICAR'].astype(str).str.upper().str.strip()

print("Filtrando productos exclusivos de Sicar...")
# 6. ENCONTRAR LOS QUE NO ESTÁN: 
# Nos quedamos con los que NO coinciden en código numérico Y TAMPOCO coinciden en descripción
sicar_no_en_pm1 = sicar_df[
    (~sicar_df['Clave_sicar_str'].isin(pm1_codigos_num)) & 
    (~sicar_df['Desc_upper'].isin(pm1_desc_alpha))
]

# 7. Seleccionar columnas y guardar
out_cols = ['Clave_sicar', 'Descripcion_SICAR', 'Precio_sicar']
res_final = sicar_no_en_pm1[out_cols]

ruta_salida = r"C:\LAB3\3\DATA\Sicar_No_En_PuntoMi.csv"
res_final.to_csv(ruta_salida, index=False, encoding="utf-8-sig")

print(f"Proceso terminado. Se encontraron {len(res_final)} artículos en SICAR que no existen en el sistema PM1.")
print(f"El archivo se ha guardado exitosamente en: {ruta_salida}")

Leyendo el archivo Sicar...


C:\Users\alvar\anaconda3\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


Total de productos extraídos de Sicar: 6498
Leyendo el archivo pm1...
Filtrando productos exclusivos de Sicar...
Proceso terminado. Se encontraron 3269 artículos en SICAR que no existen en el sistema PM1.
El archivo se ha guardado exitosamente en: C:\LAB3\3\DATA\Sicar_No_En_PuntoMi.csv


##### Para el nuevo sistema, la columna de "clave" debe de ser la misma que la de "codigo de barras" por lo que se hizo el filtrado de los productos con discrepancia para luego hacer su limpieza. 

In [3]:

ruta_cat = r"C:\LAB3\3\DATA\cat.xlsx"

import pandas as pd

ruta_cat = r"C:\LAB3\3\DATA\cat.xlsx"

print(f"Leyendo el archivo: {ruta_cat}...")
df_cat = pd.read_excel(ruta_cat)


df_cat['Clave_limpia'] = df_cat.iloc[:, 0].astype(str).str.strip()
df_cat['Codigo_limpio'] = df_cat.iloc[:, 3].astype(str).str.strip()

df_diferentes = df_cat[
    (df_cat['Clave_limpia'] != df_cat['Codigo_limpio']) & 
    (df_cat['Clave_limpia'] != 'nan') & 
    (df_cat['Codigo_limpio'] != 'nan')
]

df_resultado = df_diferentes.drop(columns=['Clave_limpia', 'Codigo_limpio'])

ruta_salida = r"Claves_Vs_Codigos_Diferentes.csv"
df_resultado.to_csv(ruta_salida, index=False, encoding="utf-8-sig")

print(f"¡Proceso terminado!")
print(f"Se encontraron {len(df_resultado)} productos donde la Columna A y la Columna D NO son iguales.")
print(f"El reporte se guardó como: {ruta_salida}")

Leyendo el archivo: C:\LAB3\3\DATA\cat.xlsx...
¡Proceso terminado!
Se encontraron 166 productos donde la Columna A y la Columna D NO son iguales.
El reporte se guardó como: Claves_Vs_Codigos_Diferentes.csv
